## Nemotron-SWE-v1

In [7]:
from datasets import load_from_disk
import random
from collections import defaultdict

N_TRAIN_TRAJ = 1000
N_TEST_TRAJ = 100
SEED = 42
path = "/var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/nemotron"

ds = load_from_disk(path)["r2e_gym"]

def split_unique_uuid_rows(ds, n_train=N_TRAIN_TRAJ, n_test=N_TEST_TRAJ, seed=SEED):
    # создаём train и test выборки
    # мы решили, что лучше пусть не будет одинаковых uuid не только в разных выборках,
    # но и внутри каждой из обеих выборок (train и test),
    # чтобы модель не затачивалась сильнее под какую-то конкретную проблему (запрос) от пользователя
    
    rnd = random.Random(seed)

    # group row indices by uuid
    uuid_to_indices = defaultdict(list)
    for idx, u in enumerate(ds["uuid"]):
        uuid_to_indices[u].append(idx)

    unique_uuids = list(uuid_to_indices.keys()) # .keys() словаря всегда уникальны
    rnd.shuffle(unique_uuids)

    need = n_train + n_test
    if len(unique_uuids) < need:
        raise ValueError(f"Need {need} unique uuid values, but got only {len(unique_uuids)}")

    train_uuids = set(unique_uuids[:n_train])
    test_uuids = set(unique_uuids[n_train:n_train + n_test])

    # choose exactly one row per uuid
    train_indices = [rnd.choice(uuid_to_indices[u]) for u in train_uuids]
    test_indices = [rnd.choice(uuid_to_indices[u]) for u in test_uuids]

    train_ds = ds.select(train_indices)
    test_ds = ds.select(test_indices)

    # safety checks
    assert len(set(train_ds["uuid"])) == len(train_ds)
    assert len(set(test_ds["uuid"])) == len(test_ds)
    # isdisjoint — проверяет, не имеют ли множества общих элементов (множества непересекаются)
    # (мы хотим, чтобы траектории с одинаковыми запросами пользователя 
    # (определяются по идентификатору запроса пользователя в столбце "uuid") не попали в разные выборки:
    # чтобы не получилось так, что модель обучилась на конкретной задаче пользователя,
    # а затем такая же задача попалась ей в тестовой выборке)
    assert set(train_ds["uuid"]).isdisjoint(set(test_ds["uuid"]))

    return train_ds, test_ds


train_raw, test_raw = split_unique_uuid_rows(ds, N_TRAIN_TRAJ, N_TEST_TRAJ, SEED)
print(len(train_raw), len(test_raw))
print(len(set(train_raw["uuid"])), len(set(test_raw["uuid"])))


# сохраним преобразованный датасет
train_raw.save_to_disk("/var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/nemotron_train_raw")
test_raw.save_to_disk("/var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/nemotron_test_raw")

#from datasets import load_from_disk
#train_raw = load_from_disk("/var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/nemotron_train_raw")
#test_raw = load_from_disk("/var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/nemotron_test_raw")

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

1000 100
1000 100


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [8]:
!ls -lh /var/essdata/DN_1/storage/home/ttn/data/swe_trajectories

total 68K
drwxr-xr-x 3 root root 4.0K Feb 25 00:21 nebius
drwxr-xr-x 3 root root 4.0K Feb 24 22:39 nemotron
drwxr-xr-x 2 root root  32K Apr 14 13:14 nemotron_chunked
drwxr-xr-x 2 root root 4.0K Apr 24 17:49 nemotron_prepared
drwxr-xr-x 2 root root  16K Apr 26 23:31 nemotron_test_chunked
drwxr-xr-x 2 root root 4.0K May  1 00:16 nemotron_test_raw
drwxr-xr-x 2 root root  16K Apr 26 23:31 nemotron_train_chunked
drwxr-xr-x 2 root root 4.0K May  1 00:16 nemotron_train_raw
drwxr-xr-x 3 root root 4.0K Feb 24 22:40 openhands
drwxr-xr-x 5 root root 4.0K Feb 25 00:26 smith


In [ ]:
#!rm -rf /var/essdata/DN_1/storage/home/ttn/data/swe_trajectories/НАЗВАНИЕ